# **Hybrid LSTM + RF dengan Hyperparameter Tuning**

Here’s the LSTM + Random Forest Hybrid Model with Hyperparameter Tuning using GridSearchCV for the best accuracy. 🚀

This approach combines deep learning (LSTM) to extract meaningful features from text and Random Forest (RF) to classify sentiments.


**Steps in the Code**

1.   Dataset balancing antara: Positive, Neutral & Nagative.
2.   Label Encoding, Text Processing, Tokenisasi.
3.   Split dataset: Data Training dan Data Test
4.   LSTM Model extracts feature embeddings.
5.   Extracted Features (LSTM last hidden state) are used as input to Random Forest.
6.   Hyperparameter Tuning for Random Forest using GridSearchCV.
7.   Evaluasi model & prediction example


**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

**Load & Balance Dataset**

In [ ]:
# Load dataset
df = pd.read_csv('/content/sample_data/Tweets.csv', encoding='utf-8')
df = df[['airline_sentiment', 'text']].dropna()

In [ ]:
# Balancing Number of Dataset (negative, neutral, positive)
# Separate sentiment classes
df_negative = df[df['airline_sentiment'] == 'negative']
df_neutral = df[df['airline_sentiment'] == 'neutral']
df_positive = df[df['airline_sentiment'] == 'positive']

# Oversample neutral and positive classes to match negative
df_neutral_oversampled = resample(df_neutral, replace=True, n_samples=len(df_negative), random_state=42)
df_positive_oversampled = resample(df_positive, replace=True, n_samples=len(df_negative), random_state=42)

# Combine to create a balanced dataset
df_balanced = pd.concat([df_negative, df_neutral_oversampled, df_positive_oversampled]) # Use this for oversampling
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle dataset

print(df_balanced['airline_sentiment'].value_counts())  # Check balance

airline_sentiment
neutral     9178
positive    9178
negative    9178
Name: count, dtype: int64


**Encode Labels & Clean Text**

In [ ]:
# Label Encoding
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_balanced['label'] = df_balanced['airline_sentiment'].map(label_map)

# Label Encoding
#y = LabelEncoder().fit_transform(df_balanced['airline_sentiment'])

# Text Preprocessing
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower().strip()  # Remove special characters & lowercase
    return text

df_balanced['text'] = df_balanced['text'].apply(clean_text)

**Tokenization & Padding**

In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=13386)  # Limit vocabulary size
tokenizer.fit_on_texts(df_balanced['text'])
sequences = tokenizer.texts_to_sequences(df_balanced['text'])
X = pad_sequences(sequences, maxlen=100)

# Labels
y = df_balanced['label'].values

**Train-Test Split**

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Build & Train LSTM Model**

In [ ]:
from tensorflow.keras.utils import to_categorical

# Convert labels to one-hot encoding
y_train_oh = to_categorical(y_train, num_classes=3)
y_test_oh = to_categorical(y_test, num_classes=3)

# Define LSTM Model
input_layer = Input(shape=(100,))
embedding_layer = Embedding(input_dim=13386, output_dim=50)(input_layer)
lstm_layer = Bidirectional(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.01)))(embedding_layer)
dropout_layer = Dropout(0.3)(lstm_layer)
dense_layer = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(dropout_layer)
feature_output = Dense(32, activation='relu')(dense_layer)

# **Final Softmax Layer for Classification**
output_layer = Dense(3, activation='softmax')(feature_output)

# Corrected Model
lstm_model = Model(inputs=input_layer, outputs=output_layer)
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train The Model
lstm_model.fit(X_train, y_train_oh, validation_split=0.2, epochs=10, batch_size=64)

Epoch 1/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 109s 371ms/step - accuracy: 0.4711 - loss: 1.9569 - val_accuracy: 0.6911 - val_loss: 0.7646
Epoch 2/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 143s 377ms/step - accuracy: 0.7603 - loss: 0.6556 - val_accuracy: 0.8030 - val_loss: 0.5879
Epoch 3/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 141s 374ms/step - accuracy: 0.8526 - loss: 0.4645 - val_accuracy: 0.8143 - val_loss: 0.5792
Epoch 4/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 104s 377ms/step - accuracy: 0.8910 - loss: 0.3815 - val_accuracy: 0.8425 - val_loss: 0.5088
Epoch 5/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 142s 375ms/step - accuracy: 0.9061 - loss: 0.3353 - val_accuracy: 0.8504 - val_loss: 0.5035
Epoch 6/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 104s 378ms/step - accuracy: 0.9193 - loss: 0.2955 - val_accuracy: 0.8470 - val_loss: 0.5120
Epoch 7/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 141s 375ms/step - accuracy: 0.9396 - loss: 0.2493 - val_accuracy: 0.8650 - val_loss: 0.4726
Epoch 8/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 141s 370ms/step - accuracy: 0.9434 -

**Extract LSTM Features & Normalize for RF**

In [12]:
# Extract Features
X_train_features = lstm_model.predict(X_train)
X_test_features = lstm_model.predict(X_test)

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features)
X_test_features = scaler.transform(X_test_features)

689/689 ━━━━━━━━━━━━━━━━━━━━ 65s 93ms/step
173/173 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step


**Train & Tune Random Forest**

In [13]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Train Random Forest with Grid Search
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train_features, y_train)

# Best Model
best_rf = grid_search.best_estimator_

# Evaluate
y_pred = best_rf.predict(X_test_features)
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid LSTM + RF Accuracy: {accuracy:.4f}')

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Hybrid LSTM + RF Accuracy: 0.9188


**Prediction Function**

In [16]:
def predict_sentiment(text, lstm_model, rf_model, tokenizer, scaler, max_length=100):
    """Preprocess input text and predict sentiment using Hybrid LSTM + RF."""
    text_clean = clean_text(text)
    sequence = tokenizer.texts_to_sequences([text_clean])
    padded_sequence = pad_sequences(sequence, maxlen=max_length)

    # Extract features using LSTM
    features = lstm_model.predict_on_batch(padded_sequence)

    # Normalize features before RF prediction
    features = scaler.transform(features)

    # Predict sentiment with RF using probabilities
    sentiment_probs = rf_model.predict_proba(features)[0]
    sentiment_label = np.argmax(sentiment_probs)  # Select highest probability class

    return {0: 'negative', 1: 'neutral', 2: 'positive'}[sentiment_label]

**Test Predictions**

In [17]:
examples = [
    "I love flying with this airline! Their service is good.",
    "I love flying with this airline! Their service is amazing.",
    "The flight was okay, nothing special but not bad."
]

for text in examples:
    sentiment = predict_sentiment(text, lstm_model, best_rf, tokenizer, scaler)
    print(f"Text: {text}\nPredicted Sentiment: {sentiment}\n")

Text: I love flying with this airline! Their service is good.
Predicted Sentiment: positive

Text: I love flying with this airline! Their service is amazing.
Predicted Sentiment: positive

Text: The flight was okay, nothing special but not bad.
Predicted Sentiment: negative

